In [24]:

# AYARLAR VE VERİ YÜKLEME
import os
import random

random.seed(42)    

veri_setleri = {
    "veri_seti_1": "../data/veri_seti_1",   # klasör sınıflandırmalı (bird / drone klasörleri)
    "veri_seti_2": "../data/veri_seti_2",   # etiket dosyaları (.txt) ayrı klasörde
    "veri_seti_3": "../data/veri_seti_3",   # YOLOv8 formatı (train / valid / test)
}
resim_uzantilari = (".png", ".jpg", ".jpeg")

resimler = {}    # her veri setinin resim yollarının listesi
for isim, yol in veri_setleri.items():
    resimler[isim] = []
    if not os.path.exists(yol):
        print("UYARI: klasör bulunamadı", yol)
        continue
    for root, dirs, files in os.walk(yol):
        for file in files:
            if file.lower().endswith(resim_uzantilari):
                resimler[isim].append(os.path.join(root, file))
    print(isim, "->", len(resimler[isim]), "resim")

print("\n SINIF ADI DOSYALARI ")
# SINIF ADI DOSYALARINI GÖSTER
for isim, yol in veri_setleri.items():
    for root, dirs, files in os.walk(yol):
        for file in files:
            if file.lower() in ("classes.txt", "data.yaml", "data.yml"):
                print(f" {isim} / {file} ")
                dosya = open(os.path.join(root, file), encoding="utf-8")
                print(dosya.read())
                dosya.close()

print("\n ETİKET DOSYALARI KONTROLÜ ")
# ETİKET DOSYALARINI BUL
etiket_yolu = {}    # resim yolu, etiket (.txt) dosyasının yolu (yoksa None)

for isim in veri_setleri:
    bulunan = 0
    for resim in resimler[isim]:
        ayni_klasor = os.path.splitext(resim)[0] + ".txt"                       
        labels_klasoru = ayni_klasor.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)
        
        # Veri Seti 2'nin klasör yapısı için özel düzeltme:
        ozel_birds = ayni_klasor.replace("images_birds", "labels_birds")
        ozel_drones = ayni_klasor.replace("images_drones", "labels_drones")

        if os.path.exists(ayni_klasor):
            etiket_yolu[resim] = ayni_klasor
            bulunan += 1
        elif os.path.exists(labels_klasoru):
            etiket_yolu[resim] = labels_klasoru
            bulunan += 1
        elif os.path.exists(ozel_birds):
            etiket_yolu[resim] = ozel_birds
            bulunan += 1
        elif os.path.exists(ozel_drones):
            etiket_yolu[resim] = ozel_drones
            bulunan += 1
        else:
            etiket_yolu[resim] = None
            
    print(isim, "->", len(resimler[isim]), "resimden", bulunan, "tanesinin etiket dosyası bulundu")
def kutulari_oku(etiket_dosyasi):
    """Etiket dosyasını okur. Her nesne için (sinif_no, x, y, genislik, yukseklik) listesi döndürür."""
    kutular = []
    dosya = open(etiket_dosyasi, encoding="utf-8")
    satirlar = dosya.read().splitlines()
    dosya.close()
    for satir in satirlar:
        parcalar = satir.split()
        if len(parcalar) == 5:                      # normal kutu etiketi
            kutular.append((int(float(parcalar[0])), float(parcalar[1]), float(parcalar[2]),
                            float(parcalar[3]), float(parcalar[4])))
        elif len(parcalar) > 5:                     # çokgen etiketi: köşe noktalarından kutu hesapla
            sayilar = parcalar[1:]
            xler = []
            yler = []
            for i in range(0, len(sayilar) - 1, 2):
                xler.append(float(sayilar[i]))
                yler.append(float(sayilar[i + 1]))
            kutular.append((int(float(parcalar[0])), (min(xler) + max(xler)) / 2, (min(yler) + max(yler)) / 2,
                            max(xler) - min(xler), max(yler) - min(yler)))
    return kutular

veri_seti_1 -> 4104 resim
veri_seti_2 -> 17936 resim
veri_seti_3 -> 6029 resim

 SINIF ADI DOSYALARI 
 veri_seti_3 / data.yaml 
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['Bird', 'Drone']

roboflow:
  workspace: drone-vs-bird-4idpt
  project: drone-vs-bird-v3
  version: 3
  license: CC BY 4.0
  url: https://universe.roboflow.com/drone-vs-bird-4idpt/drone-vs-bird-v3/dataset/3

 ETİKET DOSYALARI KONTROLÜ 
veri_seti_1 -> 4104 resimden 0 tanesinin etiket dosyası bulundu
veri_seti_2 -> 17936 resimden 17936 tanesinin etiket dosyası bulundu
veri_seti_3 -> 6029 resimden 6029 tanesinin etiket dosyası bulundu


In [28]:
# VERİ SETİ GENEL ÖZELLİKLERİ
from PIL import Image

# Etiket numarası -> sınıf adı (0: Bird, 1: Drone olarak doğrulandı)
sinif_adlari = {
    "veri_seti_2": {0: "bird", 1: "drone"},
    "veri_seti_3": {0: "bird", 1: "drone"},
}

sonuc = {}              # raporda ve grafiklerde kullanacağım değerler
sinif_resimleri = {}    # veri seti -> {"bird": [yollar], "drone": [yollar]}
bozuk_listesi = []      # bozuk dosyaların yolları

print(" VERİ SETİ GENEL ÖZELLİKLERİ")

for isim, yol in veri_setleri.items():
    liste = resimler[isim]
    if len(liste) == 0:
        continue

    kus_sayisi = 0
    drone_sayisi = 0
    sinifsiz_sayisi = 0
    kutu_sayisi = 0
    egitim_sayisi = 0
    dogrulama_sayisi = 0
    test_sayisi = 0
    etiketli = 0
    etiketsiz = 0
    bozuk = 0
    formatlar = []
    min_gen = 100000
    max_gen = 0
    min_yuk = 100000
    max_yuk = 0
    sinif_resimleri[isim] = {"bird": [], "drone": []}

    for resim in liste:
        goreli = os.path.dirname(resim)[len(yol):].lower()

        # Format
        uzanti = os.path.splitext(resim)[1].lower().replace(".", "")
        if uzanti not in formatlar:
            formatlar.append(uzanti)

        # Eğitim / doğrulama / test
        if "train" in goreli:
            egitim_sayisi += 1
        elif "valid" in goreli or "val" in goreli:
            dogrulama_sayisi += 1
        elif "test" in goreli:
            test_sayisi += 1

        # Sınıf: önce klasör adından
        sinif = None
        if "bird" in goreli:
            sinif = "bird"
        elif "drone" in goreli:
            sinif = "drone"

        # Etiket dosyası
        if etiket_yolu[resim] is not None:
            kutular = kutulari_oku(etiket_yolu[resim])
            if len(kutular) > 0:
                etiketli += 1
                kutu_sayisi += len(kutular)
                if sinif is None:                              
                    sinif_no = kutular[0][0]
                    if isim in sinif_adlari and sinif_no in sinif_adlari[isim]:
                        sinif = sinif_adlari[isim][sinif_no]
            else:
                etiketsiz += 1                                 
        else:
            etiketsiz += 1                                     

        # Sınıf sayaçları
        if sinif == "bird":
            kus_sayisi += 1
            sinif_resimleri[isim]["bird"].append(resim)
        elif sinif == "drone":
            drone_sayisi += 1
            sinif_resimleri[isim]["drone"].append(resim)
        else:
            sinifsiz_sayisi += 1

        # Boyut ve bozuk dosya kontrolü
        try:
            with Image.open(resim) as img:
                gen = img.width
                yuk = img.height
                img.verify()                                   
            if gen < min_gen: min_gen = gen
            if gen > max_gen: max_gen = gen
            if yuk < min_yuk: min_yuk = yuk
            if yuk > max_yuk: max_yuk = yuk
        except Exception:
            bozuk += 1
            bozuk_listesi.append(resim)

    sonuc[isim] = {"toplam": len(liste), "bird": kus_sayisi, "drone": drone_sayisi, "sinifsiz": sinifsiz_sayisi,
                   "egitim": egitim_sayisi, "dogrulama": dogrulama_sayisi, "test": test_sayisi,
                   "bozuk": bozuk, "etiketli": etiketli, "etiketsiz": etiketsiz, "kutu": kutu_sayisi}

    print(f"\n {isim} \n")
    print(f"Toplam Veri Sayısı: {len(liste)}")
    print(f"Veri Formatı: {', '.join(formatlar)}")
    print(f"Veri Boyutu: genişlik {min_gen}–{max_gen}, yükseklik {min_yuk}–{max_yuk}")
    if egitim_sayisi > 0 or dogrulama_sayisi > 0 or test_sayisi > 0:
        print(f"Dağılım: Eğitim: {egitim_sayisi} | Doğrulama: {dogrulama_sayisi} | Test: {test_sayisi}")
    else:
        print("Dağılım: Tek klasör (Eğitim/Doğrulama/Test ayrımı yapılmamış)")
    print(f"Sınıf Sayısı: {int(kus_sayisi > 0) + int(drone_sayisi > 0)}")
    print(f"Sınıflara Göre Veri Dağılımı: Kuş (Bird): {kus_sayisi} | İHA (Drone): {drone_sayisi} | Belirlenemeyen: {sinifsiz_sayisi}")
    print(f"Eksik/Bozuk Veri: {bozuk} bozuk görsel, {etiketsiz} resimde etiket yok veya boş")
    if etiketli == 0:
        print("Etiket Durumu: Etiket dosyası yok (sınıf klasör adından alınıyor)")
    else:
        print(f"Etiket Durumu: {etiketli} resimde etiket var, toplam {kutu_sayisi} nesne (kutu)")

 VERİ SETİ GENEL ÖZELLİKLERİ

 veri_seti_1 

Toplam Veri Sayısı: 4104
Veri Formatı: jpg, jpeg, png
Veri Boyutu: genişlik 163–4633, yükseklik 114–3089
Dağılım: Tek klasör (Eğitim/Doğrulama/Test ayrımı yapılmamış)
Sınıf Sayısı: 2
Sınıflara Göre Veri Dağılımı: Kuş (Bird): 1605 | İHA (Drone): 2499 | Belirlenemeyen: 0
Eksik/Bozuk Veri: 0 bozuk görsel, 4104 resimde etiket yok veya boş
Etiket Durumu: Etiket dosyası yok (sınıf klasör adından alınıyor)

 veri_seti_2 

Toplam Veri Sayısı: 17936
Veri Formatı: png
Veri Boyutu: genişlik 640–640, yükseklik 640–640
Dağılım: Tek klasör (Eğitim/Doğrulama/Test ayrımı yapılmamış)
Sınıf Sayısı: 2
Sınıflara Göre Veri Dağılımı: Kuş (Bird): 8936 | İHA (Drone): 9000 | Belirlenemeyen: 0
Eksik/Bozuk Veri: 0 bozuk görsel, 0 resimde etiket yok veya boş
Etiket Durumu: 17936 resimde etiket var, toplam 20664 nesne (kutu)

 veri_seti_3 

Toplam Veri Sayısı: 6029
Veri Formatı: jpg
Veri Boyutu: genişlik 640–640, yükseklik 640–640
Dağılım: Eğitim: 5397 | Doğrulama: 379 